# Multi-seed deep models and significance tests (ISO-NE load)

1. Trains LSTM, cross-attention Transformer and PatchTST with **5 seeds** each (sets B and B+) on ISO-NE.
2. Reports mean ± standard deviation over seeds.
3. Diebold–Mariano tests per zone: each deep model vs **gradient boosting** and vs **Chronos-2**, for every seed.
4. Chronos-2 vs gradient boosting.

Run the cells from top to bottom. Each finished training run is saved in `~/Downloads/seed_results/cache`,
so if you stop, just run everything again and it continues where it stopped.
At the end, send `~/Downloads/seed_results.zip` to Claude.

In [ ]:
# ===================================================================
# STEP 1 — SETTINGS (edit the paths if your files are elsewhere)
# ===================================================================
import os, time, copy, random, warnings, pickle
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)

HOME = Path.home()
DATA = HOME / "Downloads" / "open-forecasting-framework-v2" / "data" / "raw"
ISONE_FILES = [DATA / "isone" / f"{y}_smd_hourly.xlsx" for y in (2023, 2024, 2025)]
PROJECT_RESULTS = HOME / "Downloads" / "open-forecast-complete" / "results"   # your earlier full run
OUT = HOME / "Downloads" / "seed_results"      # every result of this notebook goes here

SEEDS = [42, 1, 2, 3, 4]                       # 5 training seeds per model
MODELS = ["lstm", "cross_attention_transformer", "patchtst"]
MODES = ["weather_time_history", "weather_time_history_weekly"]   # B and B+
RUN_DEEP = True
DEVICE = "mps"           # Apple-silicon GPU; change to "cpu" if you get MPS errors
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
SEED = 42
H = 24                   # forecast horizon (hours)

(OUT / "cache").mkdir(parents=True, exist_ok=True)
for f in ISONE_FILES:
    print("OK     " if f.exists() else "MISSING", f)

def cached(name, compute):
    """Run compute() once and save the result; later runs load it from disk."""
    path = OUT / "cache" / f"{name}.pkl"
    if path.exists():
        with open(path, "rb") as fh:
            print(f"[loaded from cache] {name}")
            return pickle.load(fh)
    start = time.time()
    result = compute()
    with open(path, "wb") as fh:
        pickle.dump(result, fh)
    print(f"[computed in {(time.time() - start) / 60:.1f} min] {name}")
    return result

In [ ]:
# ===================================================================
# STEP 2 — LOAD ISO NEW ENGLAND ZONAL LOAD AND WEATHER (main dataset)
# Declared daylight-saving policy:
#  * Hr_End is hour-ending local time. 2024-2025 files follow the true clock
#    (23-h day in March, 25-h day in November with "02X").
#  * 2023 uses a legacy 24-rows-per-day convention: the synthetic spring HE02 is
#    dropped; the averaged autumn HE02 is kept and the second hour interpolated.
#  * Timestamps: hour-beginning on a fixed UTC-05:00 clock (continuous index).
# ===================================================================
ZONES = ["ME", "NH", "VT", "CT", "RI", "SEMA", "WCMA", "NEMA"]
LOCAL_TZ, FIXED_OFFSET = "America/New_York", pd.Timedelta(hours=-5)

def load_isone_zone(path, zone):
    frame = pd.read_excel(path, sheet_name=zone, usecols=["Date", "Hr_End", "RT_Demand", "Dry_Bulb", "Dew_Point"])
    frame["h"] = frame["Hr_End"].astype(str).str.strip()
    frame["Date"] = pd.to_datetime(frame["Date"])
    counts = {"spring_dropped": 0, "autumn_interpolated": 0}
    days = []
    for day, g in frame.groupby("Date", sort=True):
        g = g.copy()
        start = pd.Timestamp(day).tz_localize(LOCAL_TZ)
        end = pd.Timestamp(day + pd.Timedelta(days=1)).tz_localize(LOCAL_TZ)
        real = int((end - start) / pd.Timedelta(hours=1))
        if len(g) == 24 and real == 23:
            g = g[g["h"].str.lstrip("0") != "2"]; counts["spring_dropped"] += 1
        if len(g) == 24 and real == 25:
            pos = g.index[g["h"].str.lstrip("0") == "2"][0]
            gap = g.loc[[pos]].copy(); gap[["RT_Demand", "Dry_Bulb", "Dew_Point"]] = np.nan
            g = pd.concat([g.loc[:pos], gap, g.loc[pos + 1:]]); counts["autumn_interpolated"] += 1
        assert len(g) == real, f"{path.name} {zone} {day.date()}: {len(g)} rows for {real} hours"
        utc = start.tz_convert("UTC") + pd.to_timedelta(np.arange(real), unit="h")
        g.index = (utc + FIXED_OFFSET).tz_localize(None)
        days.append(g[["RT_Demand", "Dry_Bulb", "Dew_Point"]])
    return pd.concat(days), counts

def load_isone_load():
    columns, audit = {}, {}
    for zone in ZONES:
        parts = []
        for path in ISONE_FILES:
            data, c = load_isone_zone(path, zone); parts.append(data)
            for k, v in c.items(): audit[k] = audit.get(k, 0) + v
        z = pd.concat(parts)
        columns[zone] = z["RT_Demand"]
        columns[f"{zone}_Dry_Bulb"] = z["Dry_Bulb"]; columns[f"{zone}_Dew_Point"] = z["Dew_Point"]
    hourly = pd.DataFrame(columns)
    hourly = hourly.reindex(pd.date_range(hourly.index.min(), hourly.index.max(), freq="h"))
    audit["cells_interpolated"] = int(hourly.isna().sum().sum())
    return hourly.interpolate(limit=2), audit

isone, isone_audit = cached("data_isone_load", load_isone_load)
ISONE_WEATHER = [f"{z}_{v}" for z in ZONES for v in ("Dry_Bulb", "Dew_Point")]
print("Shape:", isone.shape, "| from", isone.index.min(), "to", isone.index.max())
print("Daylight-saving audit (all zones):", isone_audit)

In [ ]:
# ===================================================================
# STEP 3 — INFORMATION SETS AND HORIZON-PURGED SPLIT
#  A  weather_time                : weather at t+H (declared proxy) + calendar of t+H
#  B  weather_time_history        : A + load lags at t-1,2,3,6,12,24 h + 3 h / 24 h rolling means
#  B+ weather_time_history_weekly : B + load at t-144 h and t-168 h
# ===================================================================
RUN_MODES = MODES
MODES = ["weather_time", "weather_time_history", "weather_time_history_weekly"]
MODE_NAME = {"weather_time": "A", "weather_time_history": "B", "weather_time_history_weekly": "B+"}
LAGS, ROLLS, WEEKLY = [1, 2, 3, 6, 12, 24], [3, 24], [144, 168]

def calendar(index):
    return pd.DataFrame({"hour_sin": np.sin(2 * np.pi * index.hour / 24), "hour_cos": np.cos(2 * np.pi * index.hour / 24),
                         "dow_sin": np.sin(2 * np.pi * index.dayofweek / 7), "dow_cos": np.cos(2 * np.pi * index.dayofweek / 7),
                         "is_weekend": (index.dayofweek >= 5).astype(float)}, index=index)

def tabular(hourly, targets, weather, mode):
    x = hourly[weather].shift(-H).add_prefix("target_weather_")
    cal = calendar(hourly.index + pd.Timedelta(hours=H)); cal.index = hourly.index
    x = pd.concat([x, cal], axis=1)
    if mode != "weather_time":
        for z in targets:
            for lag in LAGS: x[f"{z}_lag_{lag}h"] = hourly[z].shift(lag)
            for w in ROLLS: x[f"{z}_roll_{w}h"] = hourly[z].rolling(w).mean()
            if mode == "weather_time_history_weekly":
                for lag in WEEKLY: x[f"{z}_weekly_{lag}h"] = hourly[z].shift(lag)
    y = hourly[targets].shift(-H)
    ok = x.notna().all(axis=1) & y.notna().all(axis=1)
    return x.loc[ok], y.loc[ok]

def purged_split(index, train_frac, val_frac):
    n = len(index); tr = int(n * train_frac); ve = tr + int(n * val_frac)
    return {"train": index[:tr], "validation": index[tr + H:ve], "test": index[ve + H:]}

def prepare(hourly, targets, weather, train_frac, val_frac):
    frames = {m: tabular(hourly, targets, weather, m) for m in MODES}
    common = frames[MODES[0]][0].index
    for m in MODES[1:]: common = common.intersection(frames[m][0].index)
    frames = {m: (x.loc[common], y.loc[common]) for m, (x, y) in frames.items()}
    return frames, purged_split(common, train_frac, val_frac)

iso_frames, iso_split = prepare(isone, ZONES, ISONE_WEATHER, 0.5, 0.1667)
for name, sp in [("ISO-NE", iso_split)]:
    print(f"\n{name} split (forecast origins, {H}-h purge between partitions):")
    print(pd.DataFrame({k: [len(v), v.min(), v.max()] for k, v in sp.items()}, index=["origins", "first", "last"]).T)
    assert sp["train"].max() + pd.Timedelta(hours=H) < sp["validation"].min()
    assert sp["validation"].max() + pd.Timedelta(hours=H) < sp["test"].min()
print("Purge check passed. Features per mode (ISO-NE):", {MODE_NAME[m]: f.shape[1] for m, (f, _) in iso_frames.items()})

In [ ]:
# ===================================================================
# STEP 4 — METRICS AND DIEBOLD-MARIANO TEST
# ===================================================================
def scores(actual, pred, zones, model, mode):
    rows = []
    for j, z in enumerate(zones + ["aggregate"]):
        a = actual if z == "aggregate" else actual[:, [j]]; p = pred if z == "aggregate" else pred[:, [j]]
        e = a - p
        rows.append({"model": model, "set": MODE_NAME.get(mode, mode), "scope": z,
                     "RMSE": np.sqrt((e ** 2).mean()), "MAE": np.abs(e).mean(),
                     "nRMSE_%": 100 * np.sqrt((e ** 2).mean()) / np.abs(a).mean(),
                     "MAPE_%": 100 * np.mean(np.abs(e) / np.abs(a))})
    return pd.DataFrame(rows)

def dm_test(e_model, e_bench, h=H):
    """DM on squared errors, Newey-West variance with h-1 lags, HLN correction. stat < 0: model better."""
    d = e_model ** 2 - e_bench ** 2; n = len(d); dc = d - d.mean()
    gamma = [np.dot(dc[k:], dc[:n - k]) / n for k in range(h)]
    var = max(gamma[0] + 2 * sum((1 - k / h) * gamma[k] for k in range(1, h)), 1e-12)
    stat = d.mean() / np.sqrt(var / n) * np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
    return stat, 2 * stats.t.sf(abs(stat), df=n - 1)

RESULTS = {}   # (model, mode, seed) -> {"pred": array, "actual": array}

In [ ]:
# ===================================================================
# STEP 5 — DEEP MODEL DEFINITIONS (same as in the paper)
# ===================================================================
if RUN_DEEP:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset

    def sequences(hourly, targets, weather, mode, split, lookback=24):
        exo = pd.concat([hourly[weather].shift(-H), calendar(hourly.index + pd.Timedelta(hours=H)).set_index(hourly.index)], axis=1).to_numpy(np.float32)
        tv = hourly[targets].to_numpy(np.float32)
        weekly = [hourly[targets].shift(l).to_numpy(np.float32) for l in WEEKLY]
        pos = {t: i for i, t in enumerate(hourly.index)}
        parts = {}
        for name, idx in split.items():
            xs, ys = [], []
            for t in idx:
                end = pos[t]; s = end - lookback + 1
                per_zone = []
                for j in range(len(targets)):
                    block = [exo[s:end + 1]]
                    if mode != "weather_time":
                        block.append(tv[s:end + 1, j:j + 1])
                        if mode == "weather_time_history_weekly":
                            block += [w[s:end + 1, j:j + 1] for w in weekly]
                    per_zone.append(np.concatenate(block, axis=1))
                xs.append(np.stack(per_zone)); ys.append(tv[end + H])
            parts[name] = (np.stack(xs).astype(np.float32), np.stack(ys).astype(np.float32))
        return parts

    class LSTMNet(nn.Module):
        def __init__(s, f, z, hidden=128, layers=2, dropout=0.1):
            super().__init__()
            s.lstm = nn.LSTM(f * z, hidden, layers, batch_first=True, dropout=dropout, bidirectional=True)
            s.head = nn.Sequential(nn.Linear(2 * hidden, hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, z))
        def forward(s, x):
            out, _ = s.lstm(x.permute(0, 2, 1, 3).flatten(2)); return s.head(out[:, -1])

    class CrossBlock(nn.Module):
        def __init__(s, d, heads, dropout):
            super().__init__()
            s.att = nn.MultiheadAttention(d, heads, dropout=dropout, batch_first=True)
            s.n1, s.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
            s.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Dropout(dropout), nn.Linear(4 * d, d))
        def forward(s, q, c):
            a, _ = s.att(q, c, c, need_weights=False); x = s.n1(q + a); return s.n2(x + s.ff(x))

    class CrossAttentionNet(nn.Module):
        """Each zone keeps its own stream and attends to the mean of the other zones."""
        def __init__(s, f, z, d=64, heads=4, layers=2, dropout=0.1):
            super().__init__(); s.z = z
            s.proj = nn.Linear(f, d)
            layer = nn.TransformerEncoderLayer(d, heads, 4 * d, dropout, activation="gelu", batch_first=True, norm_first=True)
            s.self_enc = nn.ModuleList([nn.TransformerEncoder(copy.deepcopy(layer), 1) for _ in range(z)])
            s.blocks = nn.ModuleList([CrossBlock(d, heads, dropout) for _ in range(layers)])
            s.head = nn.Sequential(nn.LayerNorm(d * z), nn.Linear(d * z, z))
        def forward(s, x):
            st = [s.self_enc[i](s.proj(x[:, i])) for i in range(s.z)]
            for b in s.blocks:
                st = [b(st[i], torch.stack([o for k, o in enumerate(st) if k != i]).mean(0)) for i in range(s.z)]
            return s.head(torch.cat([t.mean(1) for t in st], 1))

    class PatchTSTNet(nn.Module):
        def __init__(s, f, z, d=64, heads=4, layers=2, patch=4, stride=2, dropout=0.1):
            super().__init__(); s.patch, s.stride = patch, stride
            s.proj = nn.Linear(patch * f, d)
            s.enc = nn.TransformerEncoder(nn.TransformerEncoderLayer(d, heads, 4 * d, dropout, activation="gelu", batch_first=True, norm_first=True), layers)
            s.head = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, 1))
        def forward(s, x):
            b, z, l, f = x.shape
            p = x.unfold(2, s.patch, s.stride).permute(0, 1, 2, 4, 3).reshape(b * z, -1, s.patch * f)
            return s.head(s.enc(s.proj(p)).mean(1)).reshape(b, z)

    DEEP_MODELS = {"lstm": LSTMNet, "cross_attention_transformer": CrossAttentionNet, "patchtst": PatchTSTNet}
    torch_device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if DEVICE == "mps" and torch.backends.mps.is_available() else "cpu"))

    def fit_deep(hourly, targets, weather, split, model_name, mode, epochs=60, patience=10, seed=SEED):
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        parts = sequences(hourly, targets, weather, mode, split)
        (xtr, ytr), (xva, yva), (xte, yte) = parts["train"], parts["validation"], parts["test"]
        # the 24-h lookback of the earliest origins can reach before the first weekly lag: drop those samples
        keep = lambda x, y: ~(np.isnan(x).any(axis=(1, 2, 3)) | np.isnan(y).any(axis=1))
        k = keep(xtr, ytr); xtr, ytr = xtr[k], ytr[k]
        k = keep(xva, yva); xva, yva = xva[k], yva[k]
        assert not np.isnan(xte).any(), "test inputs contain NaN"
        xm, xs = xtr.reshape(-1, xtr.shape[-1]).mean(0), xtr.reshape(-1, xtr.shape[-1]).std(0) + 1e-8
        ym, ys = ytr.mean(0), ytr.std(0) + 1e-8
        nx = lambda a: ((a - xm) / xs).astype(np.float32); ny = lambda a: ((a - ym) / ys).astype(np.float32)
        net = DEEP_MODELS[model_name](xtr.shape[-1], ytr.shape[-1]).to(torch_device)
        opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4); loss_fn = nn.MSELoss()
        loader = DataLoader(TensorDataset(torch.from_numpy(nx(xtr)), torch.from_numpy(ny(ytr))), batch_size=64, shuffle=True)
        vx, vy = torch.from_numpy(nx(xva)).to(torch_device), torch.from_numpy(ny(yva)).to(torch_device)
        best, best_state, wait = float("inf"), copy.deepcopy(net.state_dict()), 0
        for epoch in range(epochs):
            net.train()
            for bx, by in loader:
                opt.zero_grad(); loss = loss_fn(net(bx.to(torch_device)), by.to(torch_device)); loss.backward()
                torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step()
            net.eval()
            with torch.no_grad(): v = loss_fn(net(vx), vy).item()
            if v < best - 1e-7: best, best_state, wait = v, copy.deepcopy(net.state_dict()), 0
            else: wait += 1
            if wait >= patience: break
        net.load_state_dict(best_state); net.eval()
        with torch.no_grad(): pred = net(torch.from_numpy(nx(xte)).to(torch_device)).cpu().numpy() * ys + ym
        print(f"   {model_name} {MODE_NAME[mode]}: stopped after {epoch + 1} epochs")
        return yte, pred

In [ ]:
# ===================================================================
# STEP 6 — TRAIN EVERY MODEL WITH 5 SEEDS ON ISO-NE (resumable)
# Each finished run is saved in seed_results/cache. If you stop the
# notebook, run it again from the top: finished runs load in seconds.
# ===================================================================
rows = []
for model_name in MODELS:
    for mode in RUN_MODES:
        for seed in SEEDS:
            name = f"isone_{model_name}_{mode}_seed{seed}"
            actual, pred = cached(name, lambda: fit_deep(isone, ZONES, ISONE_WEATHER, iso_split, model_name, mode, seed=seed))
            t = scores(actual, pred, ZONES, model_name, mode); t["seed"] = seed
            RESULTS[(model_name, mode, seed)] = {"actual": actual, "pred": pred}
            rows.append(t)
            agg = t[t.scope == "aggregate"].iloc[0]
            print(f"   {model_name} {MODE_NAME[mode]} seed {seed}: RMSE {agg.RMSE:.1f} MW, nRMSE {agg['nRMSE_%']:.2f}%", flush=True)
seed_table = pd.concat(rows)
seed_table.to_csv(OUT / "seed_metrics_all.csv", index=False)

In [ ]:
# ===================================================================
# STEP 7 — SEED SUMMARY: mean, standard deviation, min and max over seeds
# ===================================================================
agg = seed_table[seed_table.scope == "aggregate"]
summary = (agg.groupby(["model", "set"])
              .agg(seeds=("seed", "nunique"), RMSE_mean=("RMSE", "mean"), RMSE_sd=("RMSE", "std"),
                   RMSE_min=("RMSE", "min"), RMSE_max=("RMSE", "max"), nRMSE_mean=("nRMSE_%", "mean"))
              .round(2))
summary.to_csv(OUT / "seed_summary.csv")
display(summary)
print("Gradient boosting (set B) for comparison: 102.2 MW")

In [ ]:
# ===================================================================
# STEP 8 — READ GRADIENT BOOSTING AND CHRONOS-2 FORECASTS FROM YOUR EARLIER RUN
# and check they are on exactly the same forecast origins and actual values.
# ===================================================================
def read_predictions(path, model, mode):
    p = pd.read_csv(path)
    p = p[(p.split == "test") & (p.model == model) & (p.feature_mode == mode)]
    act_col = "actual" if "actual" in p.columns else "actual_watts"
    pred_col = "predicted" if "predicted" in p.columns else "predicted_watts"
    a = p.pivot(index="forecast_origin", columns="zone", values=act_col)[ZONES].to_numpy()
    f = p.pivot(index="forecast_origin", columns="zone", values=pred_col)[ZONES].to_numpy()
    return a, f

OTHERS = {}
tree_file = PROJECT_RESULTS / "isone_trees" / "predictions.csv"
found_file = PROJECT_RESULTS / "isone_foundation" / "predictions.csv"
for label, path, model, mode in [("gradient boosting (B)", tree_file, "hist_gradient_boosting", "weather_time_history"),
                                 ("Chronos-2 (zero-shot)", found_file, "chronos2", "weather_time_history")]:
    if not path.exists():
        print("MISSING", path); continue
    a, f = read_predictions(path, model, mode)
    ref = RESULTS[(MODELS[0], RUN_MODES[0], SEEDS[0])]["actual"]
    assert a.shape == ref.shape and np.allclose(a, ref, atol=0.01), f"{label}: forecast origins do not match"
    OTHERS[label] = f
    print(f"OK  {label}: {a.shape[0]} forecast origins, identical actual values")

In [ ]:
# ===================================================================
# STEP 9 — DIEBOLD-MARIANO TESTS PER ZONE (squared error, HLN, Newey-West)
# For every seed: deep model vs gradient boosting, and deep model vs Chronos-2.
# stat < 0 and p < 0.05 means the deep model is significantly more accurate.
# ===================================================================
dm_rows = []
for (model_name, mode, seed), r in RESULTS.items():
    if not isinstance(seed, int):
        continue
    for label, other in OTHERS.items():
        for j, z in enumerate(ZONES):
            e_model = r["pred"][:, j] - r["actual"][:, j]
            e_other = other[:, j] - r["actual"][:, j]
            stat, p = dm_test(e_model, e_other)
            dm_rows.append({"model": model_name, "set": MODE_NAME[mode], "seed": seed, "versus": label, "zone": z,
                            "rmse_ratio": np.sqrt((e_model ** 2).mean() / (e_other ** 2).mean()), "dm": stat, "p": p})
dm = pd.DataFrame(dm_rows)
dm.to_csv(OUT / "dm_seeds_per_zone.csv", index=False)
dm["better"] = (dm.dm < 0) & (dm.p < 0.05)
dm["worse"] = (dm.dm > 0) & (dm.p < 0.05)
dm_summary = (dm.groupby(["model", "set", "versus", "seed"])[["better", "worse"]].sum()
                .groupby(["model", "set", "versus"])
                .agg(zones_better_min=("better", "min"), zones_better_max=("better", "max"),
                     zones_worse_min=("worse", "min"), zones_worse_max=("worse", "max")))
dm_summary.to_csv(OUT / "dm_seeds_summary.csv")
print("Number of the 8 zones where the deep model is significantly better / worse (range over seeds):")
display(dm_summary)

In [ ]:
# ===================================================================
# STEP 10 — CHRONOS-2 vs GRADIENT BOOSTING (no seeds: both are deterministic)
# ===================================================================
if len(OTHERS) == 2:
    a = RESULTS[(MODELS[0], RUN_MODES[0], SEEDS[0])]["actual"]
    c2, gb = OTHERS["Chronos-2 (zero-shot)"], OTHERS["gradient boosting (B)"]
    rows = []
    for j, z in enumerate(ZONES):
        stat, p = dm_test(c2[:, j] - a[:, j], gb[:, j] - a[:, j])
        rows.append({"zone": z, "rmse_ratio_chronos2_vs_gb": np.sqrt(((c2[:, j] - a[:, j]) ** 2).mean() / ((gb[:, j] - a[:, j]) ** 2).mean()),
                     "dm": stat, "p": p})
    c2_vs_gb = pd.DataFrame(rows).round(4)
    c2_vs_gb.to_csv(OUT / "dm_chronos2_vs_gb.csv", index=False)
    display(c2_vs_gb)

In [ ]:
# ===================================================================
# STEP 11 — ZIP THE RESULTS FOR CLAUDE (small CSV files only)
# ===================================================================
import zipfile
zip_path = HOME / "Downloads" / "seed_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in OUT.glob("*.csv"):
        z.write(f, f.name)
print("Send this file to Claude:", zip_path)